# Multimodal Emotion Detection
This notebook demonstrates using a unified model for emotion detection in both text and images.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the multimodal model
model_path = 'multimodal/MULTIMODAL_detection_model_hf.h5'
multimodal_model = load_model(model_path)

# Load face classifier
face_classifier = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

# Emotion labels
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

## Model Summary

In [ ]:
print("Multimodal Emotion Detection Model Summary:")
print("="*50)
multimodal_model.summary()

In [ ]:
# Model layer details
print('Model Layers:')
for i, layer in enumerate(multimodal_model.layers):
    print(f'{i+1:2d}. {layer.name:20s} - {layer.__class__.__name__:15s} - {layer.output_shape}')

## Text Emotion Detection

In [ ]:
def predict_text_emotion(text):
    """Predict emotion from text using Hugging Face model"""
    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")
    result = classifier(text)
    
    # Map to our emotion labels
    emotion_map = {'ANGER': 'Angry', 'DISGUST': 'Disgust', 'FEAR': 'Fear', 
                  'JOY': 'Happy', 'NEUTRAL': 'Neutral', 'SADNESS': 'Sad', 
                  'SURPRISE': 'Surprise'}
    
    emotions = {emotion_map.get(item['label'], item['label']): item['score'] for item in result}
    return emotions

# Test text emotion detection
sample_texts = [
    "I am so happy today!",
    "This makes me really angry.",
    "I feel sad and lonely.",
    "What a wonderful surprise!"
]

print("Text Emotion Detection Results:")
print("="*40)

for text in sample_texts:
    emotions = predict_text_emotion(text)
    dominant_emotion = max(emotions, key=emotions.get)
    confidence = emotions[dominant_emotion]
    
    print(f"\nText: '{text}'")
    print(f"Dominant Emotion: {dominant_emotion} ({confidence:.3f})")
    
    # Visualization
    plt.figure(figsize=(10, 4))
    emotions_df = pd.DataFrame(list(emotions.items()), columns=['Emotion', 'Probability'])
    emotions_df = emotions_df.sort_values('Probability', ascending=True)
    
    plt.barh(emotions_df['Emotion'], emotions_df['Probability'])
    plt.xlabel('Probability')
    plt.title(f"Emotion Distribution for: '{text}'")
    plt.tight_layout()
    plt.show()

## Image Emotion Detection

In [ ]:
def predict_image_emotion(image_path):
    """Predict emotion from image using the multimodal model"""
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_classifier.detectMultiScale(gray, 1.3, 5)
    
    if len(faces) == 0:
        return None, image
    
    results = []
    annotated_image = image.copy()
    
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_gray = cv2.resize(roi_gray, (48, 48), interpolation=cv2.INTER_AREA)
        roi = roi_gray.astype('float') / 255.0
        roi = img_to_array(roi)
        roi = np.expand_dims(roi, axis=0)
        
        # Predict emotion
        prediction = multimodal_model.predict(roi)[0]
        emotions = {emotion_labels[i]: prediction[i] for i in range(len(emotion_labels))}
        results.append(emotions)
        
        # Annotate image
        dominant_emotion = max(emotions, key=emotions.get)
        confidence = emotions[dominant_emotion]
        
        cv2.rectangle(annotated_image, (x, y), (x+w, y+h), (0, 255, 255), 2)
        cv2.putText(annotated_image, f'{dominant_emotion}: {confidence:.2f}', 
                   (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    return results, annotated_image

print("Image Emotion Detection Function Ready")
print("Usage: results, annotated_img = predict_image_emotion('path/to/image.jpg')")

## Model Analysis

In [ ]:
print("Model Analysis:")
print("="*20)
print(f"Total parameters: {multimodal_model.count_params():,}")
print(f"Number of layers: {len(multimodal_model.layers)}")
print(f"Input shape: {multimodal_model.input_shape}")
print(f"Output shape: {multimodal_model.output_shape}")

# Layer details
print("\nLayer Details:")
for i, layer in enumerate(multimodal_model.layers):
    params = layer.count_params()
    print(f"{i+1:2d}. {layer.name:20s} - {layer.__class__.__name__:15s} - {str(layer.output_shape):20s} - {params:,} params")

## Interactive Demo

In [ ]:
def interactive_text_demo():
    print("Interactive Text Emotion Detection")
    print("Type 'quit' to exit")
    
    while True:
        text = input("\nEnter text to analyze: ")
        if text.lower() == 'quit':
            break
        
        emotions = predict_text_emotion(text)
        dominant_emotion = max(emotions, key=emotions.get)
        confidence = emotions[dominant_emotion]
        
        print(f"Dominant Emotion: {dominant_emotion} ({confidence:.3f})")
        print("All emotions:")
        for emotion, prob in sorted(emotions.items(), key=lambda x: x[1], reverse=True):
            print(f"  {emotion}: {prob:.3f}")

# Uncomment to run: interactive_text_demo()